In [1]:
from datasets import load_dataset
dataset = load_dataset('glue', 'sst2')
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [2]:
print(dataset['test'][2])

{'sentence': 'by the end of no such thing the audience , like beatrice , has a watchful affection for the monster .', 'label': -1, 'idx': 2}


In [3]:
from transformers import AutoTokenizer
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        truncation=True,
        max_length=128,
    )

tokenized = dataset.map(tokenize_function, batched=True)
print(tokenized['train'].column_names)
print(tokenized['train'][0].keys())

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']
dict_keys(['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'])


In [4]:
tokenized = tokenized.remove_columns(['sentence', 'idx'])
tokenized = tokenized.rename_column('label', 'labels')
print(tokenized['train'].column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [5]:
!pip install -q evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print(model)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [7]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels)
    return {**acc, **f1}

training_args = TrainingArguments(
    output_dir='./sst2-bert',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=True,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.058871,0.354414,0.913991,0.918033
2,0.081024,0.286341,0.913991,0.917310
3,0.057795,0.329453,0.923165,0.925638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6315, training_loss=0.07468424487095075, metrics={'train_runtime': 611.3469, 'train_samples_per_second': 330.495, 'train_steps_per_second': 10.33, 'total_flos': 4176656240000220.0, 'train_loss': 0.07468424487095075, 'epoch': 3.0})

In [8]:
print(trainer.state.log_history)

[{'loss': 0.1505147647857666, 'grad_norm': 2.0877952575683594, 'learning_rate': 5.224274406332454e-06, 'epoch': 0.047505938242280284, 'step': 100}, {'loss': 0.12218324661254883, 'grad_norm': 8.847384452819824, 'learning_rate': 1.050131926121372e-05, 'epoch': 0.09501187648456057, 'step': 200}, {'loss': 0.12696178436279296, 'grad_norm': 3.7361857891082764, 'learning_rate': 1.577836411609499e-05, 'epoch': 0.14251781472684086, 'step': 300}, {'loss': 0.07947601318359375, 'grad_norm': 3.405210256576538, 'learning_rate': 1.9932614555256068e-05, 'epoch': 0.19002375296912113, 'step': 400}, {'loss': 0.09288016319274903, 'grad_norm': 0.2739999294281006, 'learning_rate': 1.959568733153639e-05, 'epoch': 0.2375296912114014, 'step': 500}, {'loss': 0.06841966152191162, 'grad_norm': 3.0956969261169434, 'learning_rate': 1.9258760107816713e-05, 'epoch': 0.2850356294536817, 'step': 600}, {'loss': 0.07329174518585205, 'grad_norm': 2.0866734981536865, 'learning_rate': 1.8921832884097035e-05, 'epoch': 0.3325

In [10]:
import pandas as pd
preds_output = trainer.predict(tokenized['validation'])
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(labels, preds, target_names=['negative', 'positive']))
cm = confusion_matrix(labels, preds)
df = pd.DataFrame(
    cm,
    index=['Actual Neg', 'Actual Pos'],
    columns=['Pred Neg', 'Pred Pos']
)
print(df)


              precision    recall  f1-score   support

    negative       0.93      0.91      0.92       428
    positive       0.91      0.94      0.93       444

    accuracy                           0.92       872
   macro avg       0.92      0.92      0.92       872
weighted avg       0.92      0.92      0.92       872

            Pred Neg  Pred Pos
Actual Neg       388        40
Actual Pos        27       417


In [14]:
wrong_indices = np.where(preds != labels)[0]
for i in wrong_indices[:10]:
    original = dataset['validation'][int(i)]
    print(f"Text: {original['sentence']}")
    print(f"True: {original['label']}, Pred: {preds[i]}")
print('-'*10)

Text: we root for ( clara and paul ) , even like them , though perhaps it 's an emotion closer to pity . 
True: 1, Pred: 0
Text: holden caulfield did it better . 
True: 0, Pred: 1
Text: it offers little beyond the momentary joys of pretty and weightless intellectual entertainment . 
True: 0, Pred: 1
Text: this one is definitely one to skip , even for horror movie fanatics . 
True: 0, Pred: 1
Text: you wo n't like roger , but you will quickly recognize him . 
True: 0, Pred: 1
Text: if steven soderbergh 's ` solaris ' is a failure it is a glorious failure . 
True: 1, Pred: 0
Text: this riveting world war ii moral suspense story deals with the shadow side of american culture : racial prejudice in its ugly and diverse forms . 
True: 0, Pred: 1
Text: hilariously inept and ridiculous . 
True: 1, Pred: 0
Text: sam mendes has become valedictorian at the school for soft landings and easy ways out . 
True: 0, Pred: 1
Text: every nanosecond of the the new guy reminds you that you could be doing s

In [15]:
print(tokenized['validation'][0])
print(dataset['validation'][0])

{'labels': 1, 'input_ids': [101, 2009, 1005, 1055, 1037, 11951, 1998, 2411, 12473, 4990, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
{'sentence': "it 's a charming and often affecting journey . ", 'label': 1, 'idx': 0}
